In [1]:
import sys
import os

# Convert relative path to absolute, append once
sys.path.append(os.path.abspath("Vintern-1B-v3_5/vintern_local"))

In [2]:
# import os
# print(os.listdir("Vintern-1B-v3_5/vintern_local"))

In [3]:
# Now import WITHOUT prefix
# from processing_vinternvl import VinternVLProcessor
# from modeling_intern_vit import InternVLChatModel
# from transformers import AutoConfig, BlipProcessor, BlipForConditionalGeneration
# from PIL import Image
# import torch

In [4]:
import torch
from transformers import AutoModel, AutoTokenizer

model = AutoModel.from_pretrained(
    "5CD-AI/Vintern-1B-v3_5",
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
    use_flash_attn=False,
).eval().cuda()

tokenizer = AutoTokenizer.from_pretrained("5CD-AI/Vintern-1B-v3_5", trust_remote_code=True, use_fast=False)


c:\Users\admin\anaconda3\envs\smoke-detect-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\admin\anaconda3\envs\smoke-detect-env\lib\site-packages\timm\models\layers\__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


FlashAttention2 is not installed.


Sliding Window Attention is enabled but not implemented for `eager`; unexpected results may be encountered.


In [5]:
from PIL import Image
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def build_transform(input_size):
    transform = T.Compose([
        T.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ])
    return transform

def load_image(image_file, input_size=448):
    image = Image.open(image_file).convert('RGB')
    transform = build_transform(input_size=input_size)
    image = transform(image)
    return image.unsqueeze(0)  # Add batch dimension


In [6]:
# Load YOLOv5n from torch hub
yolo_model = torch.hub.load('ultralytics/yolov5', 'yolov5n', pretrained=True)
yolo_model.conf = 0.1   # Lower confidence threshold
yolo_model.iou = 0.45   # NMS IoU threshold


Using cache found in C:\Users\admin/.cache\torch\hub\ultralytics_yolov5_master
YOLOv5  2025-4-30 Python-3.10.16 torch-2.1.0+cu118 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)

100%|██████████| 3.87M/3.87M [00:00<00:00, 7.91MB/s]

Fusing layers... 
YOLOv5n summary: 213 layers, 1867405 parameters, 0 gradients, 4.5 GFLOPs
Adding AutoShape... 


In [7]:
import cv2
from PIL import Image
import numpy as np

def detect_and_crop_person(image_path):
    img_bgr = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    results = yolo_model(img_rgb, size=960)
    boxes = results.xyxy[0].cpu().numpy()

    person_boxes = [box for box in boxes if int(box[5]) == 0]

    if person_boxes:
        x1, y1, x2, y2, conf, cls = sorted(person_boxes, key=lambda x: x[4], reverse=True)[0]
        x1, y1, x2, y2 = map(int, [x1, y1, x2, y2])
        crop = img_bgr[y1:y2, x1:x2]
        cv2.imwrite("temp_crop.jpg", crop)
        return "temp_crop.jpg", True
    else:
        return image_path, False


In [8]:
def detect_smoking(image_path):
    def ask_vintern(img_path):
        image_tensor = load_image(img_path).to(torch.bfloat16).cuda()
        question = (
            "<image>\n"
            "Does the image show any person holding a cigarette or joint, or exhaling smoke?"
        )


        generation_config = dict(
            max_new_tokens=1024,
            do_sample=False,
            num_beams=3,
            repetition_penalty=2.5
        )
        response, _ = model.chat(
            tokenizer, image_tensor, question,
            generation_config,
            history=None,
            return_history=True
        )
        return response

    # Step 1: Ask Vintern on the full image first
    print("Checking full image with Vintern...")
    response = ask_vintern(image_path)
    response_lower = response.lower().strip()
    print(f'User: <image>\nIs there any sign of smoking in this image?\nAssistant: {response}')

    # Step 2: Keyword analysis
    # Define keywords
    positive_keywords = ["yes"]
    negative_keywords = ["no"]

    # Analyze first sentence
    first_sentence = response_lower.split(".")[0]

    # Case 1: Strong negative with no smoking cues
    if any(neg in first_sentence for neg in negative_keywords) and not any(pos in first_sentence for pos in positive_keywords):
        print("No smoking behavior detected.")
        return False, None

    # Case 2: Conflicting cues in first sentence
    if any(neg in first_sentence for neg in negative_keywords) and any(pos in first_sentence for pos in positive_keywords):
        print("Conflicting cues in first sentence. Proceeding with full response check...")

    # Case 3: Smoking behavior indicated somewhere in the response
    if any(pos in response_lower for pos in positive_keywords):
        print("Smoking behavior detected. Proceeding to YOLO person crop for further analysis.")
        cropped_path, used_crop = detect_and_crop_person(image_path)
        return True, cropped_path

    # Final fallback
    print("No smoking behavior detected.")
    return False, None



In [9]:
image_tensor = load_image(r"C:\Users\admin\OneDrive\Desktop\smoking-detection\dataset\images\train\000158_jpg.rf.d21cab839d5b7d52a49f5cb9788cfa23.jpg").to(torch.bfloat16).cuda()
question = '<image>\nIs the man holding a cigarette or joint in the image?'

generation_config = dict(max_new_tokens=1024, do_sample=False, num_beams=3, repetition_penalty=2.5)

response, history = model.chat(tokenizer, image_tensor, question, generation_config, history=None, return_history=True)
print(f'User: {question}\nAssistant: {response}')



Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


User: <image>
Is the man holding a cigarette or joint in the image?
Assistant: The man is holding a cigarette in the image.


In [10]:
image_path = r"C:\Users\admin\OneDrive\Desktop\smoking-detection\dataset\images\train\000253_jpg.rf.3be13eaf90fa2a8ca4d30888447d6542.jpg"
detect_smoking(image_path)



Checking full image with Vintern...


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


User: <image>
Is there any sign of smoking in this image?
Assistant: Yes, the image shows a person holding a cigarette and exhaling smoke.
Smoking behavior detected. Proceeding to YOLO person crop for further analysis.


(True, 'temp_crop.jpg')